In [1]:
# ==========================================
# CELL 1 — Install dependency (Fabric Notebook cell)
# ==========================================
%pip install yfinance --quiet


# ==========================================
# CELL 2 — Imports & Config
# ==========================================
import yfinance as yf
import pandas as pd
from datetime import datetime
import logging
import os

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Bursa Malaysia Banking Sector Tickers
TICKERS = [
    "1155.KL",  # Maybank
    "1295.KL",  # Public Bank
    "1023.KL",  # CIMB
    "1066.KL",  # RHB
    "5258.KL"   # BIMB
]

END_DATE = datetime.now().strftime('%Y-%m-%d')
START_DATE = "2024-01-01"


# ==========================================
# CELL 3 — Extraction function (unchanged logic)
# ==========================================
def fetch_raw_data(ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
    """Fetches raw historical stock data using yf.download."""
    logging.info(f"Fetching raw data for {ticker} from {start_date} to {end_date}...")
    try:
        df = yf.download(
            tickers=ticker,
            start=start_date,
            end=end_date,
            auto_adjust=False,
            progress=False
        )

        if df.empty:
            logging.warning(f"No data found for ticker {ticker}.")
            return pd.DataFrame()

        df.reset_index(inplace=True)

        # Safeguard for MultiIndex columns in newer yfinance versions
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [col[0] for col in df.columns.values]

        df.columns = [str(col).lower().replace(" ", "_") for col in df.columns]
        return df

    except Exception as e:
        logging.error(f"Error fetching data for {ticker}: {e}")
        return pd.DataFrame()


def add_bronze_metadata(df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    """Adds ingestion metadata to the raw dataframe."""
    if df.empty:
        return df

    df['ingestion_timestamp'] = datetime.now()
    df['source_system'] = 'yahoo_finance'
    df['ticker_symbol'] = ticker
    df['exchange'] = 'Bursa Malaysia'
    return df


# ==========================================
# CELL 4 — Run extraction for all tickers, combine into ONE dataframe
# ==========================================
all_frames = []
success_count = 0

for ticker in TICKERS:
    raw_df = fetch_raw_data(ticker, START_DATE, END_DATE)

    if not raw_df.empty:
        bronze_df = add_bronze_metadata(raw_df, ticker)
        all_frames.append(bronze_df)
        success_count += 1
    else:
        logging.warning(f"⚠️ Skipped {ticker} due to empty data.")

logging.info(f"=== Fetched {success_count}/{len(TICKERS)} tickers successfully. ===")

bronze_combined = pd.concat(all_frames, ignore_index=True)
bronze_combined.head()


# ==========================================
# CELL 5 — Write to Lakehouse as a managed Delta table
#           (replaces save_to_bronze() local Parquet/CSV writes)
# ==========================================
# NOTE: make sure your MalaysiaBankLakehouse is attached to this notebook first
# (top-right "Add Lakehouse" button), otherwise spark.createDataFrame().write
# below will fail with "no default lakehouse" errors.

# ingestion_timestamp needs to be a proper timestamp type for Spark/Delta
bronze_combined['ingestion_timestamp'] = pd.to_datetime(bronze_combined['ingestion_timestamp'])
bronze_combined['date'] = pd.to_datetime(bronze_combined['date'])

spark_df = spark.createDataFrame(bronze_combined)

spark_df.write.format("delta").mode("overwrite").saveAsTable("bronze_bursa_banks")

logging.info(f"✅ Saved {bronze_combined.shape[0]} rows -> Lakehouse table: bronze_bursa_banks")

# ==========================================
# CELL 5b — Save a CSV copy into Lakehouse Files section
# ==========================================

files_path = "/lakehouse/default/Files/bronze"
os.makedirs(files_path, exist_ok=True)

csv_path = f"{files_path}/bronze_bursa_banks.csv"
bronze_combined.to_csv(csv_path, index=False)

logging.info(f"✅ Saved CSV -> {csv_path}")

# ==========================================
# CELL 6 — Quick verification
# ==========================================
spark.sql("SELECT ticker_symbol, COUNT(*) as row_count FROM bronze_bursa_banks GROUP BY ticker_symbol").show()

StatementMeta(, 8c74ce7b-90bf-4f34-9c11-39d7a8ac6214, 8, Finished, Available, Finished, False)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nni 3.0 requires filelock<3.12, but you have filelock 3.13.1 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
+-------------+---------+
|ticker_symbol|row_count|
+-------------+---------+
|      1295.KL|      659|
|      1023.KL|      659|
|      1066.KL|      659|
|      1155.KL|      659|
|      5258.KL|      659|
+-------------+---------+




2026-09-04 03:29:44,357 - INFO - Fetching raw data for 1155.KL from 2024-01-01 to 2026-09-04...
2026-09-04 03:29:45,394 - INFO - Fetching raw data for 1295.KL from 2024-01-01 to 2026-09-04...
2026-09-04 03:29:45,703 - INFO - Fetching raw data for 1023.KL from 2024-01-01 to 2026-09-04...
2026-09-04 03:29:46,032 - INFO - Fetching raw data for 1066.KL from 2024-01-01 to 2026-09-04...
2026-09-04 03:29:46,249 - INFO - Fetching raw data for 5258.KL from 2024-01-01 to 2026-09-04...
2026-09-04 03:29:46,482 - INFO - === Fetched 5/5 tickers successfully. ===
2026-09-04 03:30:03,488 - INFO - ✅ Saved 3295 rows -> Lakehouse table: bronze_bursa_banks
2026-09-04 03:30:03,719 - INFO - ✅ Saved CSV -> /lakehouse/default/Files/bronze/bronze_bursa_banks.csv
